# Tau-måling: Spektral koherens i språkmodeller

**Kjør på Colab T4 (gratis)**

Måler τ = r_eff / r_max fra skjulte tilstander via SVD.

Goldilocks: [exp(−γ), 1/ζ(3)] = [0.5615, 0.8319]

In [ ]:
# Installer avhengigheter
!pip install -q transformers accelerate bitsandbytes torch numpy

In [ ]:
# HuggingFace innlogging via Colab Secrets
# Gå til: venstre panel → nøkkel-ikon (Secrets) → legg til HF_TOKEN
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN lasta frå Colab Secrets.")
except Exception:
    print("Ingen HF_TOKEN funne — berre opne modellar tilgjengelege.")

In [ ]:
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import os

MODEL_NAME = "Qwen/Qwen2.5-14B"
PREDICTION = 0.35

bnb_config = BitsAndBytesConfig(load_in_4bit=True)
token = os.environ.get("HF_TOKEN")

print(f"Laster {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=token)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
    token=token,
)
model.eval()
print("Klar.")

In [ ]:
# Mål tau
resultater = {}
for label, text in TEXTS.items():
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    tau = measure_tau(outputs.hidden_states[-1])
    resultater[label] = tau
    status = ("GOLDILOCKS ★" if TAU_MIN <= tau <= TAU_MAX
              else "BELOW" if tau < TAU_MIN else "ABOVE")
    print(f"[{label:>10}]  tau = {tau:.4f}  {status}")

print(f"\nPrediksjon:  {PREDICTION:.4f}")
print(f"Målt:        {resultater['coherent']:.4f}")
print(f"Avvik:       {abs(resultater['coherent'] - PREDICTION):.4f}")
print(f"\nRekkefølge (koherent > tilfeldig > repetitivt): "
      f"{'JA ✓' if resultater['coherent'] > resultater['random'] > resultater['repetitive'] else 'NEI ✗'}")